# Exercise 5-1-2: Detecting the Use of Generative AI in a Financial Report

**The research task.** Firms write a lot of prose — risk factors, MD&A, the letters and reviews in an annual report — and since ChatGPT launched in November 2022, some of that prose is drafted or polished by a generative AI (GAI). How would you *measure* that from the outside, using only the published document?

Blankespoor, deHaan, and Li (2026, *JAR* 64(3):1189–1232) do exactly this for a large panel of U.S. filings. Their measure, **`GenScore`**, is the output of a commercial GAI detector, **GPTZero**, applied to a disclosure. Two of their findings set up this exercise:

* **The detector works in this setting.** In validation tests where the authors insert a known amount of GPT-written text into real filings, regressions on `GenScore` reliably detect GAI even when just **0.0625% of all text** is GAI-modified (2.5% of sentences, for 2.5% of firms). Prior skepticism about AI detectors largely predates this generation of tools or relies on open-source detectors with high false-positive rates.
* **Firms do use it.** They find statistically significant GAI usage in all five disclosure types they examine, with up to **4.5% of new text** GAI-written in 2024 — higher for small firms, firms with no investor-relations officer, and firms with recent major events.

**What we build here** is the measurement core of that paper, applied to a single document: BHP Group's FY2025 Annual Report (`data/BHP_Annual_report_2025.pdf`). We go from a raw PDF to a cleaned slice of narrative text, score it with GPTZero, reproduce the paper's `GenScore` definition, look at which sentences drive the score, and compare the result to the paper's benchmarks.

| Step | What we do |
| --- | --- |
| 1 | Extract and clean text from the PDF |
| 2 | Take the first ~5,000 words of narrative, following the paper's protocol |
| 3 | Score it with the GPTZero API and compute `GenScore` |
| 4 | Read `GenScore` at the sentence level |
| 5 | Interpret the number against the paper's benchmarks |


## Setup

**API key.** Create a free account at [gptzero.me](https://gptzero.me/), open the dashboard, and copy your API key into the repo-root `.env` file as a new line:

The free tier covers a few thousand words per month — enough for this exercise. GPTZero updates its detection model periodically, so every response carries a `version` string identifying the model that scored your text.

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pymupdf
import requests
from dotenv import load_dotenv

import os

load_dotenv()
GPTZERO_API_KEY = os.getenv("GPTZERO_API_KEY")
PDF_PATH = Path("../data/BHP_Annual_report_2025.pdf")

### Step 1. Clean text in a PDF

In [ ]:
def page_texts(path: Path) -> list[str]:
    """Return the plain text of every page in the PDF."""
    with pymupdf.open(path) as doc:
        return [page.get_text() for page in doc]

pages = page_texts(PDF_PATH)
print(f"{len(pages)} pages")

# A narrative page, as pymupdf sees it - note the hard line breaks and \xa0 spaces.
print(repr(pages[6][:600]))

#### 1.1 Normalise the characters and drop the page furniture

Two cleaning passes:

* **`normalise`** maps the typographic characters a detector shouldn't have to reason about — curly quotes, en/em dashes, non-breaking and thin spaces — to their plain ASCII equivalents, and rejoins words split by a hyphen at a line break.
* **`is_furniture`** flags lines that are page chrome rather than prose: the repeated running-header labels, bare page numbers, and lines of pure punctuation or digits from tables. This is deliberately a *heuristic* — getting it exactly right for an arbitrary report layout is hard, which is part of why the paper pays for a clean feed.


In [ ]:
_TRANSLATIONS = {
    "‘": "'", "’": "'", "“": '"', "”": '"',   # curly quotes
    "–": "-", "—": "-", "‑": "-", "−": "-",    # dashes, minus
    "\xa0": " ", " ": " ", " ": " ", " ": " ",
    " ": " ", " ": " ",
}

def normalise(text: str) -> str:
    """Fold typographic punctuation to ASCII and undo line-break hyphenation."""
    text = unicodedata.normalize("NFKC", text)
    for src, dst in _TRANSLATIONS.items():
        text = text.replace(src, dst)
    return re.sub(r"(\w)-\n(\w)", r"\1\2", text)          # "manage-\nment" -> "management"

_RUNNING_HEADERS = {
    "Overview", "Contents", "Governance", "Financial Statements",
    "Operating and Financial Review", "Additional Information",
    "BHP Annual Report 2025",
}

def is_furniture(line: str) -> bool:
    """True if a line is page chrome (header label, page number, table debris)."""
    s = line.strip()
    return (
        not s
        or s in _RUNNING_HEADERS
        or len(s) <= 2
        or re.fullmatch(r"[\d.,%$()\-\s]+", s) is not None
    )

full_text = normalise("\n".join(pages))
print(full_text[:600])

### Step 2. Take the first ~5,000 words of narrative

The paper scores **the first 5,000 words** of each disclosure, "rounding up to finish the last sentence" (§3). The cutoff is a cost decision — it binds for ~87% of MD&A and risk-factor filings — and it standardises the unit being scored.

The paper also scores *each disclosure type separately* (risk factors, MD&A, conference-call remarks, press releases). BHP's PDF isn't pre-segmented that way, so we take the report's first 5,000 words of continuous narrative: the Chair's and CEO's reviews and the opening of the Operating and Financial Review. That is the closest analogue to the MD&A-style narrative the paper studies. Keep the mismatch in mind when you read the number — it is a defensible slice, not the paper's exact construct.

We anchor on the Chair's review (the phrase also appears in the table of contents, so we require the "Dear Shareholders" salutation that follows the real heading), drop the furniture lines, then cut at 5,000 words and extend to the end of the sentence.


In [ ]:
def first_n_words(text: str, n: int = 5000) -> str:
    """First `n` whitespace-delimited words of `text`, extended to the next sentence end."""
    head = " ".join(text.split()[:n])
    tail = text[len(head):]
    end = re.search(r"[.!?](?=\s|$)", tail)
    return head + (tail[: end.end()] if end else "")

anchor = re.search(r"Chair's [Rr]eview\s+Dear Shareholders", full_text)
narrative = full_text[anchor.start():]

kept_lines = [ln.strip() for ln in narrative.split("\n") if not is_furniture(ln)]
narrative = re.sub(r"\s+", " ", " ".join(kept_lines))

sample = first_n_words(narrative, 5000)
print(f"{len(sample.split()):,} words | {len(sample):,} characters\n")
print(sample[:800], "...")


### Step 3. Score the text with GPTZero

GPTZero's text endpoint takes one `document` and returns document-, paragraph-, and sentence-level probabilities. The field we want is **`class_probabilities`**, which splits the document's authorship probability three ways: `human`, `ai`, and `mixed`.

The paper defines (footnote 9):

> `GenScore` = the sum of GPTZero's document-level probabilities of **GAI authorship plus mixed authorship**

i.e. `GenScore = 100 * (class_probabilities["ai"] + class_probabilities["mixed"])`, on a 0–100 scale. It is the estimated share of the document influenced by GAI, *not* a confidence that the whole thing is AI.

GPTZero caps request size, so `genscore_for_text` splits long text into chunks, scores each, and takes the **word-count-weighted average** of the chunk scores — the same aggregation you would use to roll sentence scores up to a document.


In [ ]:
GPTZERO_URL = "https://api.gptzero.me/v2/predict/text"

def gptzero_predict(document: str) -> dict:
    """Send one document to GPTZero; return its parsed `documents[0]` object."""
    response = requests.post(
        GPTZERO_URL,
        headers={
            "x-api-key": GPTZERO_API_KEY,
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        json={"document": document},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    doc = payload["documents"][0] if "documents" in payload else payload
    doc["api_version"] = payload.get("version") or doc.get("version")   # record the model version
    return doc

def gen_score(doc: dict) -> float:
    """GenScore = 100 * (P(ai) + P(mixed)) at the document level (Blankespoor et al. 2026, fn. 9)."""
    probs = doc.get("class_probabilities")
    if probs:
        return 100 * (probs.get("ai", 0.0) + probs.get("mixed", 0.0))
    return 100 * doc.get("completely_generated_prob", 0.0)   # fallback for older API responses

def genscore_for_text(text: str, max_chars: int = 45_000) -> dict:
    """Score `text` with GPTZero, chunking if needed and word-count-weighting the chunks."""
    words, chunks, buffer = text.split(), [], ""
    for word in words:
        if len(buffer) + len(word) + 1 > max_chars:
            chunks.append(buffer.strip())
            buffer = ""
        buffer += word + " "
    if buffer.strip():
        chunks.append(buffer.strip())

    docs = [gptzero_predict(chunk) for chunk in chunks]
    weights = [len(chunk.split()) for chunk in chunks]
    score = sum(gen_score(d) * w for d, w in zip(docs, weights)) / sum(weights)
    return {"genscore": score, "n_chunks": len(chunks), "documents": docs}


In [ ]:
result = genscore_for_text(sample)

# The API response is the primary data of the measurement - save it before doing anything else.
RAW_PATH = Path("../data/bhp_gptzero_raw.json")
RAW_PATH.write_text(json.dumps(result["documents"], indent=2))

doc = result["documents"][0]
print(f"GenScore                 : {result['genscore']:.2f}   (0-100 scale)")
print(f"class_probabilities      : {doc.get('class_probabilities')}")
print(f"completely_generated_prob: {doc.get('completely_generated_prob')}")
print(f"average_generated_prob   : {doc.get('average_generated_prob')}")
print(f"document_classification  : {doc.get('document_classification')}")
print(f"confidence_category      : {doc.get('confidence_category')}")
print(f"GPTZero model version    : {doc.get('api_version')}")


### Step 4. Read `GenScore` at the sentence level

GPTZero returns a `generated_prob` for every sentence, and the paper uses this sentence-level signal in its linguistic analyses (§5.3) — e.g. to contrast the readability and tone of the sentences *most* versus *least* likely to be GAI-written within the same report.

Here we just look at the distribution and at the specific sentences carrying the score.


In [ ]:
sentences = pd.concat(
    [pd.DataFrame(d.get("sentences", [])) for d in result["documents"]],
    ignore_index=True,
)
keep = [c for c in ["generated_prob", "perplexity", "highlight_sentence_for_ai", "sentence"] if c in sentences]
sentences = sentences[keep]

flagged = (sentences["generated_prob"] > 0.5).mean()
print(f"{len(sentences):,} sentences | {flagged:.1%} with generated_prob > 0.5\n")

ax = sentences["generated_prob"].plot.hist(bins=20, edgecolor="white")
ax.set_xlabel("sentence-level generated_prob")
ax.set_title("How AI-like is each sentence of BHP's narrative?")
plt.show()

sentences.sort_values("generated_prob", ascending=False).head(10)


### Step 5. Interpret the number

A single `GenScore` is only meaningful next to a reference point. The paper gives several:

| Reference (from Blankespoor et al. 2026) | `GenScore` |
| --- | --- |
| Human-written MD&A, 2018–2020 validation sample (Table 2) | 0.33 |
| Human-written risk factors, 2018–2020 (Table 2) | 0.50 |
| Full-sample mean, 2010–2024 (Table 1B) | 0.55 |
| Estimated GAI usage in MD&A in 2024 (Table 4B) | 0.71 |
| Estimated GAI usage in conference-call remarks in 2024 (Table 4B) | 1.43 |
| Estimated GAI usage in MD&A in 2024, *new text only* (Table 4D) | 2.50 |
| A filing fully rewritten by GPT-3.5 (Table 3B) | 92.7 |

The pre-ChatGPT human baseline sits near **0.3–0.5** — that is the detector's measurement floor, not zero. The paper removes it with firm × fiscal-quarter fixed effects; with a single document you cannot, so treat anything in that range as "indistinguishable from human-written."


In [ ]:
benchmarks = pd.DataFrame(
    {
        "reference": [
            "Human MD&A (Table 2)",
            "Human risk factors (Table 2)",
            "Full-sample mean (Table 1B)",
            "GAI usage, MD&A 2024 (Table 4B)",
            "GAI usage, conf. call 2024 (Table 4B)",
            "GAI usage, MD&A 2024, new text (Table 4D)",
            "Fully GPT-rewritten filing (Table 3B)",
        ],
        "genscore": [0.33, 0.50, 0.55, 0.71, 1.43, 2.50, 92.7],
    }
)
benchmarks = pd.concat(
    [benchmarks, pd.DataFrame([{"reference": "BHP FY2025 narrative (this notebook)", "genscore": result["genscore"]}])],
    ignore_index=True,
)

ax = benchmarks.set_index("reference")["genscore"].plot.barh(logx=True, figsize=(8, 4))
ax.set_xlabel("GenScore (log scale, 0-100)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

benchmarks